Vogliamo costruire il dataset dei prodotti Amazon nel campo Electronics, intersecando:
Amazon Reviews 2023 che contiene metadati + rating/recensioni
Amazon question/answer storico McAuley

In [2]:
# Se mancano pacchetti, esegui questa cella una sola volta.
%pip install -q "datasets==3.6.0" pandas pyarrow requests tqdm

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path #per gestire meglio percorsi di file e cartelle
import ast #servirà più avanti per leggere file del QA storico
import gzip #leggere e scrivere file compressi gz
import json #lavorare con json
import shutil #utile per varie operazioni, forse in futuro
from typing import Iterable, Optional #tipi
import pandas as pd #pandas per creare dataframe
import pyarrow as pa #per definire schemi tabellari
import pyarrow.parquet as pq    #per leggere metadata dei file .parquet
import requests #per scaricare da URL
from tqdm.auto import tqdm #barra di progresso
import datasets #pacchetto datasets
from datasets import load_dataset #per caricare dataset

datasets.logging.set_verbosity_error() #riduce i messaggi stampati da datasets

c:\Users\arsid\Desktop\Tesi-ecommerce\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#configurazione
CATEGORY_2023="Electronics" #modifica qua per cambiare categoria

QA_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon/qa/qa_Electronics.json.gz"
#per versione risposte multiple https://mcauleylab.ucsd.edu/public_datasets/data/amazon/qa/icdm/QA_Electronics.json.gz

In [5]:
PROJECT_ROOT=Path("..").resolve() if Path("..").joinpath("data").exists() else Path(".").resolve() #sceglie root
RAW_DIR=PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR=PROJECT_ROOT / "data" / "processed" / "electronics_intersection"

QA_RAW_PATH=RAW_DIR / "qa" / "qa_Electronics.json.gz"
QA_STRICT_PATH=RAW_DIR / "qa" / "qa_Electronics.strict.jsonl.gz"

ITEMS_OUT=PROCESSED_DIR / "items.parquet" #percorso finale dei prodotti
QA_OUT=PROCESSED_DIR / "qa.parquet" #percorso file domande/risposte
RATINGS_OUT=PROCESSED_DIR / "ratings.parquet" #percorso rating

RAW_DIR.mkdir(parents=True, exist_ok=True) #crea data/raw
(QA_RAW_PATH.parent).mkdir(parents=True, exist_ok=True) #crea padre data/raw/qa
PROCESSED_DIR.mkdir(parents=True, exist_ok=True) #crea data/processed/electronics_intersection

print("Project root:", PROJECT_ROOT)
print("Output:", PROCESSED_DIR)

Project root: C:\Users\arsid\Desktop\Tesi-ecommerce
Output: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection


In [6]:
def download_file(url:str,output_path:Path, force:bool=False) ->Path: #scarica da url in output_path
    if output_path.exists() and not force: #non riscarica
        print(f"Già presente: {output_path}")
        return output_path
    output_path.parent.mkdir(parents=True, exist_ok=True) #se non esiste cartella di destinazione la crea

    with requests.get(url, stream=True,timeout=60)as r:
        r.raise_for_status()
        total =int(r.headers.get("content-length", 0)) #dimensione
        with open(output_path, "wb") as f, tqdm( #apre file in scrittura binaria, perchè scarichi .gz e apre barra tqdm
            total=total,
            unit="B", #in byte
            unit_scale=True,
            desc=output_path.name,
        ) as pbar:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    return output_path

download_file(QA_URL, QA_RAW_PATH)

Già presente: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Electronics.json.gz


WindowsPath('C:/Users/arsid/Desktop/Tesi-ecommerce/data/raw/qa/qa_Electronics.json.gz')

In [7]:
#Convertiamo QA storico da JSONL gzip a JSON e aggiungiamo parent_asin per
#agganciarlo al dataset amazon 2023
#per fare questo l'asin storico lo usiamo come parent_asin
def iter_historical_qa(path: Path)->Iterable[dict]: #legge qa storico, restituisce dizionari python
    with gzip.open(path,"rt",encoding="utf-8", errors="replace") as f: #apre il file .gz
        for line in f:
            line =line.strip() #rimuove spazi e newline
            if line:
                yield ast.literal_eval(line) #converte riga da testo a dizionario python

qa_asins=set()
qa_rows=0


with gzip.open(QA_STRICT_PATH, "wt", encoding="utf-8") as out: #apre il file in scrittura, .jsonl.gz
    for qa_id,row in enumerate(tqdm(iter_historical_qa(QA_RAW_PATH), desc="Conversione QA")): #scorre tutti i record del qa storico
        asin=row.get("asin") #prende valore dal campo asin
        row["qa_id"] = qa_id #aggiunge id numerico alla riga qa
        row["parent_asin"] = asin #crea il campo parent_asin e ci copia l'asin

        if asin: #se asin non è vuoto, aggiunge l'asin all'insieme dei prodotti presenti nel qa
            qa_asins.add(asin)

        out.write(json.dumps(row, ensure_ascii=False) + "\n") #scrive la riga nel nuovo file jsonl
        qa_rows += 1

print("QA rows:", qa_rows)
print("Prodotti unici nel QA:", len(qa_asins))
print("File strict:", QA_STRICT_PATH)

Conversione QA: 314263it [00:12, 24340.79it/s]

QA rows: 314263
Prodotti unici nel QA: 39371
File strict: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Electronics.strict.jsonl.gz


In [8]:
#carichiamo metadati amazon reviews 2023 da Hugging Face (piattaforma dove è pubblicato il dataset)
meta_ds = load_dataset( #risultato salvato in meta_ds
    "McAuley-Lab/Amazon-Reviews-2023", #nome del dataset su HuggingFace
    f"raw_meta_{CATEGORY_2023}", #raw_meta_electronics per ora
    split="full", 
    trust_remote_code=True,
)
print(meta_ds)
print("Numero metadati:", len(meta_ds))

Dataset({
    features: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author'],
    num_rows: 1610012
})
Numero metadati: 1610012


In [10]:
def image_url(images)->Optional[str]: #prova a estrarre immagine da campo images
    if isinstance(images,dict):
        for key in ("hi_res","large","thumb"): #prova alcune chiavi
            values=images.get(key) or []
            for value in values:
                if value:
                    return value
    if isinstance(images, list):
        for img in images:
            if isinstance(img, dict):
                for key in ("large_image_url","medium_image_url","small_image_url"):
                    value=img.get(key)
                    if value:
                        return value
    return None

def tojson_string(value)->str: #converte valore in stringa json
    if value is None:
        return "null"
    return json.dumps(value, ensure_ascii=False)

def tofloat_or_none(value): #converte valore in numero decimale
    try:
        if value in (None,"","None"):
            return None
        return float(value)
    except Exception:
        return None

def get_category(row: dict) ->Optional[str]:
    main_category=row.get("category")
    if main_category not in (None,"", "None"):
        return main_category

    categories=row.get("categories")
    if isinstance(categories, list) and len(categories)>0:
        return categories[0]

    return CATEGORY_2023

def toint_or_none(value): #converte in intero
    try:
        if value in (None,"", "None"):
            return None
        return int(value)
    except Exception:
        return None

ITEM_SCHEMA = pa.schema([ #schema del file items.parquet
    ("parent_asin", pa.string()),
    ("title", pa.string()),
    ("category", pa.string()),
    ("avg_rating", pa.float64()),
    ("num_ratings", pa.int64()),
    ("price", pa.float64()),
    ("store", pa.string()),
    ("features_json", pa.string()),
    ("description_json", pa.string()),
    ("categories_json", pa.string()),
    ("details_json", pa.string()),
    ("image_url", pa.string()),
])

#scrive righe in un file parquet, rows=listadiz, path=dove salvare, schema=schema PyArrow, writer=oggetto che gestisce la scrittura Parquet
def writerows_in_parquet(rows:list[dict], path:Path, schema: pa.schema, writer:Optional[pq.ParquetWriter]):
    if not rows:
        return writer

    df=pd.DataFrame(rows) #converte lista di dizionari in un dataframe pandas
    table=pa.Table.from_pandas(df, schema=schema, preserve_index=False) #converte dataframe in tabella pyarrow

    if writer is None:
        path.parent.mkdir(parents=True, exist_ok=True) #crea cartella di output
        writer=pq.ParquetWriter(path, schema=schema, compression="zstd") #crea un writer Parquet,compresso
    writer.write_table(table)
    return writer

#dobbiamo filtrare solo elementi presenti anche nel QA storico
if ITEMS_OUT.exists(): #se items.parquet esiste già lo elimina
    ITEMS_OUT.unlink()

writer=None
buffer=[]
matched_items=0

for row in tqdm(meta_ds, desc="Filtro metadati"): #scorre tutti i prodotti nei metadati amazon 2023 electronics
    parent_asin=row.get("parent_asin")
    if parent_asin not in qa_asins:
        continue
    buffer.append({ #se prodotto è presente nel qa, lo aggiunge al buffer
        "parent_asin": parent_asin,
        "title": row.get("title"),
        "category":get_category(row),
        "avg_rating": tofloat_or_none(row.get("avg_rating")),
        "num_ratings": toint_or_none(row.get("num_ratings")),
        "price": tofloat_or_none(row.get("price")),
        "store": row.get("store"),
        "features_json": tojson_string(row.get("features")),
        "description_json": tojson_string(row.get("description")),
        "categories_json": tojson_string(row.get("categories")),
        "details_json": tojson_string(row.get("details")),
        "image_url": image_url(row.get("images")),
    })
    matched_items+=1

    if len(buffer) >= 50_000:
        writer=writerows_in_parquet(buffer, ITEMS_OUT, ITEM_SCHEMA, writer)
        buffer=[]
writer = writerows_in_parquet(buffer, ITEMS_OUT, ITEM_SCHEMA, writer) #nel caso avesse meno di 50000 el.    
if writer is not None:
    writer.close()

items_df=pd.read_parquet(ITEMS_OUT)
items_id=set(items_df["parent_asin"])

print("Prodotti QA ∩ metadata 2023:", len(items_id))
display(items_df.head())


Filtro metadati: 100%|██████████| 1610012/1610012 [02:17<00:00, 11739.26it/s]


Prodotti QA ∩ metadata 2023: 22975


,parent_asin,title,category,avg_rating,num_ratings,price,store,features_json,description_json,categories_json,details_json,image_url
0,B00CFGZAT8,Dock Audio Extender Adapter Converter Cable fo...,Electronics,NaN,NaN,NaN,iteck,[],[],"[""Electronics"", ""Computers & Accessories"", ""Ta...","""{\""Brand Name\"": \""iTeck\"", \""Item Weight\"": ...",https://m.media-amazon.com/images/I/41uTuqlHDS...
1,B00BGFTSGU,GGS Swivi HD DSLR LCD Universal Foldable Viewf...,Electronics,NaN,NaN,149.00,Swivi,[],[],"[""Electronics"", ""Camera & Photo"", ""Accessories...","""{\""Product Dimensions\"": \""3.8 x 4.3 x 6.9 in...",https://m.media-amazon.com/images/I/41BUv5PjMD...
2,B00AZOIG9I,BM Premium 2-Pack of NP-85 Batteries and Charg...,Electronics,NaN,NaN,22.99,BM Premium,"[""NP85 Li-ion Battery for FujiFilm FinePix SL2...","[""Bring your digital camera back to life with ...","[""Electronics"", ""Camera & Photo"", ""Accessories...","""{\""Package Dimensions\"": \""6.46 x 3.94 x 1.85...",https://m.media-amazon.com/images/I/71fbU-87Tx...
3,B00F361KLO,ZAGG InvisibleShield HD – EZ Apply Film Screen...,Electronics,NaN,NaN,NaN,InvisibleShield,"[""100% Clear: Independent light transmission t...","[""Exceptionally clear, unbelievably thin, and ...","[""Electronics"", ""Computers & Accessories"", ""Ta...","""{\""Standing screen display size\"": \""9.7 Inch...",https://m.media-amazon.com/images/I/71XY4kKwcX...
4,B0029L7N76,Mobile Edge ECO Laptop Messenger (Eco-Friendly...,Electronics,NaN,NaN,NaN,Mobile Edge,"[""Fits 17.3-Inch laptops. Laptop compartment d...","[""The 17.3-Inch ECO Messenger is the latest ad...","[""Electronics"", ""Computers & Accessories"", ""La...","""{\""Product Dimensions\"": \""4.25 x 18 x 13.5 i...",https://m.media-amazon.com/images/I/91JdX5IqgA...
